In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
from scipy import ndimage
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import pandas as pd
import netCDF4 as nc
import re
from skimage.measure import regionprops
from scipy.ndimage import maximum_filter
import glob
import matplotlib.ticker as mticker

In [ ]:
tbb_threshold = -52                              # Cloud Top Brightness Threshold (°C)
min_area = 50000                                 # Area becoming MCS
pixel_size = 5.0                                 # Himawari-8/9 Pixel Resolution (km)

rootpath = '/proj4/data/Himawari8/'
save_path = '/proj6/users/ma/data/output'
if not os.path.exists(save_path):
        os.makedirs(save_path)
search_pattern = os.path.join(rootpath, '*', '*.nc')
file_list = glob.glob(search_pattern)
file_list.sort()

# file_list = file_list[:24]
# print(len(file_list))

In [ ]:
def create_region_mask(lat_array, lon_array):
    
    lon_grid, lat_grid = np.meshgrid(lon_array, lat_array)
    base_condition = (
        (lat_grid >= 18) & (lat_grid <= 55) & # modify here
        (lon_grid >= 80) & (lon_grid <= 146)
    )

    final_mask = base_condition
    
    return final_mask

In [ ]:
def detect_mcs(tbb_celsius,region_mask,lat, lon,time_info):
        #dectect mcs in mask
        binary_mask = (tbb_celsius <= tbb_threshold) & region_mask
        #selected_tbb = tbb_celsius[binary_mask]
        cold_pixels = np.sum(binary_mask)
        if cold_pixels == 0:
            print("not a pixel meet the required")
            return np.zeros_like(tbb_data), []

        
        structure = ndimage.generate_binary_structure(2, 2)  # 8-连通结构
        binary_mask_closed = ndimage.binary_closing(binary_mask, structure=structure, iterations=2)
        labeled_array, num_features = ndimage.label(binary_mask_closed,structure=structure)
        
        props = regionprops(labeled_array, intensity_image=tbb_celsius)
        
        mcs_list = []  #MCS info
        valid_mask = np.zeros_like(labeled_array, dtype=int)  #MCS label
        
        current_id = 1
        
        for prop in props:
            
            #area
            area_km2 = prop.area * (pixel_size ** 2)
            if area_km2 < min_area:
                continue
                
            if prop.eccentricity > 0.95:
                continue
 
            
             #MCS center
            center_y, center_x = prop.centroid
            center_y, center_x = int(center_y), int(center_x)
            

            center_lat = lat[center_y]
            center_lon = lon[center_x]
    
            
            #center have to located in research region
            if not region_mask[center_y, center_x]:
                continue
            

            c_lat = lat[center_y]
            c_lon = lon[center_x]
            
            # document
            mcs_info = {
                'time': time_info,
                'id_in_frame': current_id,
                'area_km2': area_km2,
                'center_lat': c_lat,
                'center_lon': c_lon,
                'center_idx': (center_y, center_x),
                'min_tbb': prop.min_intensity,
                'mean_tbb': prop.mean_intensity,
                'eccentricity': prop.eccentricity
            }
            
            mcs_list.append(mcs_info)
            coords = prop.coords
            valid_mask[coords[:, 0], coords[:, 1]] = current_id
        
            current_id += 1
            
        return mcs_list, valid_mask

In [ ]:
def run_tracking(file_list):
    global_next_id = 1
    
    prev_active_tracks = {}
    
    prev_labeled_array = None
    all_tracks_data = []
    
    for t, filepath in enumerate(file_list):
        basename = os.path.basename(filepath)

        pattern = r'(\d{4})(\d{2})(\d{2})_(\d{2})(\d{2})'
        match = re.search(pattern, basename)
        year = match.group(1)   # 2016
        month = match.group(2)  # 06
        day = match.group(3)    # 06
        hour = match.group(4)   # 00
        minute = match.group(5) # 00

        time_info = f"{year}-{month}-{day} {hour}-{minute}"

        #def read_himawari_netcdf(filepath):
        #get tbb
        var_name = 'tbb_15'
        dataset = nc.Dataset(filepath, 'r')
        tbb_data = dataset.variables[var_name][:] - 273.15
        tbb_celsius = tbb_data
        #print(tbb_data)
        #get lat lon
        lat = dataset.variables['latitude'][:]
        lon = dataset.variables['longitude'][:]
        #print(tbb_data, lat, lon, time_info)
        dataset.close()
        
        current_step_mcs_list=[]
        region_mask = create_region_mask(lat, lon)
        curr_mcs_list,curr_labeled= detect_mcs(tbb_celsius,region_mask,lat, lon,time_info)
        print(f"detect {len(curr_mcs_list)} 个MCS")
        current_frame_mapping = {}

        if prev_labeled_array is None:
            for mcs in curr_mcs_list:
                local_id = mcs['id_in_frame']
                global_id = global_next_id
                global_next_id += 1
                
                current_frame_mapping[local_id] = global_id
                
                mcs['global_id'] = global_id
                mcs['status'] = 'Genesis' 
                all_tracks_data.append(mcs)
                current_step_mcs_list.append(mcs)#画图用
        else:
            dilated_prev_labels = maximum_filter(prev_labeled_array, size=11)#膨胀一下，来计算重叠
        
            for mcs in curr_mcs_list:
                local_id = mcs['id_in_frame']
                current_cloud_mask = (curr_labeled == local_id)
            
                overlap_prev_ids = dilated_prev_labels[current_cloud_mask]
            
                overlap_prev_ids = overlap_prev_ids[overlap_prev_ids != 0]
                
                if len(overlap_prev_ids) == 0:

                    global_id = global_next_id
                    global_next_id += 1
                    status = 'Genesis'
                else:
                    parent_local_id = np.argmax(np.bincount(overlap_prev_ids))
                    

                    if parent_local_id in prev_active_tracks:
                        global_id = prev_active_tracks[parent_local_id]
                        status = 'Continuation'
                    else:
                    
                        global_id = global_next_id
                        global_next_id += 1
                        status = 'Genesis (Complex)'

                #记录
                current_frame_mapping[local_id] = global_id
                mcs['global_id'] = global_id
                mcs['status'] = status
                all_tracks_data.append(mcs)
                current_step_mcs_list.append(mcs)#画图用
        prev_labeled_array = curr_labeled
        prev_active_tracks = current_frame_mapping

        #start draw     
        fig_width_px = 2100
        fig_height_px = 1350
        dpi = 150
        map_extent = [80, 145, 18, 55]
        mcs_tbb_masked = np.ma.masked_where(curr_labeled == 0, tbb_celsius)
        fig = plt.figure(figsize=(fig_width_px/dpi, fig_height_px/dpi), dpi=dpi)
        proj = ccrs.PlateCarree()
        ax = fig.add_axes([0.08, 0.15, 0.84, 0.78], projection=proj)
        ax.set_extent(map_extent, crs=proj)
        
        ax.coastlines(resolution='50m', color='gray', linewidth=0.7)
        ax.add_feature(cfeature.BORDERS, linestyle=':', edgecolor='gray')
        
        gl = ax.gridlines(crs=proj, draw_labels=True,
                      linewidth=1, color='gray', alpha=0.5, linestyle='--')# lon lan
        gl.xlocator = mticker.MultipleLocator(10)
        gl.ylocator = mticker.MultipleLocator(10)
        gl.top_labels = False    # 关掉上边
        gl.right_labels = False  # 关掉右边
        gl.xlabel_style = {'size': 10, 'color': 'black'} # 设置经度字体大小
        gl.ylabel_style = {'size': 10, 'color': 'black'} 
        
        tbb = ax.pcolormesh(lon, lat, tbb_celsius, transform=proj, cmap='gray', vmin=-80, vmax=20, alpha=0.3)
    
        mcs = ax.pcolormesh(lon, lat, mcs_tbb_masked, transform=proj, 
                             cmap='jet_r', vmin=-80, vmax=-32, zorder=2)
    
        ax.contour(lon, lat, curr_labeled, levels=[0.5], 
               colors='magenta', linewidths=0.6, transform=proj, zorder=2)
    
        for mcs_record in current_step_mcs_list:
            uid = mcs_record['global_id'] # 'global_id'命名
            c_lat = mcs_record['center_lat']
            c_lon = mcs_record['center_lon']
        # 写 ID
            ax.text(c_lon, c_lat, f"{int(uid)}", transform=proj,
                    color='black', fontsize=12, fontweight='bold',
                    ha='center', va='bottom')
        cbar_ax1 = fig.add_axes([0.15, 0.08, 0.35, 0.02]) # 左边的图例轴
        cbar_ax2 = fig.add_axes([0.55, 0.08, 0.35, 0.02]) # 右边的图例轴
        
        cb1 = plt.colorbar(tbb, cax=cbar_ax1, orientation='horizontal')
        cb1.set_label('Background TBB (°C)', fontsize=10)
        cb1.ax.tick_params(labelsize=9)

        cb2 = plt.colorbar(mcs, cax=cbar_ax2, orientation='horizontal')
        cb2.set_label('MCS TBB (°C)', fontsize=10)
        cb2.ax.tick_params(labelsize=9)
        current_time_str = str(time_info) # 或者 str(times[t])
        ax.set_title(f'MCS Detection UTC:{year}{month}{day}_{hour}{minute}',fontsize=14, fontweight='bold')
        
        ax.set_xlim(80, 145)
        ax.set_ylim(18, 55)

        ax.autoscale(False)
        #draw legend
        # legend_elements = [
        # mpatches.Patch(facecolor='gray', label='origin TBB (grayscale)'),
        # mpatches.Patch(facecolor='red', label=f'MCS regions (colored)'),
        # plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='k', 
        #            markersize=6, label='MCS center')
        # ]
        # ax.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
        output_file = os.path.join(save_path, f'MCS_pptgif_{year}{month}{day}_{hour}{minute}.png')
        fig.canvas.print_figure(output_file, 
                        dpi=dpi,
                        bbox_inches=None,
                        facecolor='white',
                        edgecolor='none')
        plt.savefig(output_file, dpi=300, bbox_inches='tight')

        plt.show()
        print(f"saved: {global_next_id}")
        plt.close(fig) 
        #end draw

    return all_tracks_data

In [ ]:
all_tracks_data = run_tracking(file_list)

#print(all_tracks_data)

# df_result = pd.DataFrame(all_tracks_data)
# output_filename = 'mcs_tracks_h8_coversea.csv'
# full_save_path = os.path.join(save_path, output_filename)
# df_result.to_csv(full_save_path, index=False, encoding='utf-8')
print('end')